In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama

## Two approaches to Guadrails

### Deterministic Guardrails
- Rule-based: regex, keyword matching, explicit checks
- ✅ Fast, predictable, cost-effective
- ❌ May miss nuanced violations

In [5]:
import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


### Model Based Guadrails

- Uses LLMs/classifiers for semantic understanding
- ✅ Catches subtle/nuanced issues
- ❌ Slower and more expensive

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [9]:
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [11]:
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""

    prompt = f"""Is the following user input safe to process?
            Reply with only 'SAFE' or 'UNSAFE'.

            Input: {text}"""
            
    result = model.invoke([{"role": "user", "content": prompt}])
    
    # --- PERBAIKAN DI SINI ---
    content = result.content
    
    # Cek apakah content berupa list (contoh: [{'type': 'text', 'text': 'SAFE'}])
    if isinstance(content, list):
        # Ambil nilai 'text' dari elemen pertama list
        if len(content) > 0 and isinstance(content[0], dict) and "text" in content[0]:
            content = content[0]["text"]
        else:
            # Fallback jika bentuk list-nya berbeda
            content = str(content[0])
            
    # Pastikan menjadi string dan hilangkan spasi/karakter newline
    return str(content).strip()

# (Sisa kode Anda di bawahnya tetap sama)
print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


## Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [33]:
model = ChatOllama(
    model="llama3.2:3b",
)

In [28]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [24]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I can't assist with creating a function call that could be used for fraudulent activities such as accessing customer information. Can I help you with something else?


In [32]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='9bc62fb7-4274-4759-98a2-725a9e83715e'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'customer_lookup', 'arguments': '{"query": "[REDACTED_EMAIL] 5100"}'}, '__gemini_function_call_thought_signatures__': {'768674b0-a4f2-48df-ab13-b3eaed629999': 'Cq0DARFNMg/JWaZYp9RjnkFr7bIupvNQbo90t56/e8e9DaRuQ7XYpKEZm1NXmPLef+qCOGSCqb1pQur/66xaV4KL2yx5exbUI2Y4EUvHYeGrI5do/sCvb2DWIMlFs9ruAVmjDhJndHAqXWwHMCyWEuWA+D0HaNghTfahEixKVburg9jXagXflrSpOGPj3rWdtFAaqjoTVACmD7orNWVXlQCHvMtgwhR1cRvXyYJNmeF1FjV3kWXWj9WA0qTWubihQNjRj+7cZFZYDLgdXZj7n95jsyY3gQmKuCXHIfllVjUNo8V/74miEKDrcuGtrto370NKqEBipe326TGDMgC0b+9U+T2M4iPLZzj+KAui8pE5AScdBmGYV/pHq2L7CnwoC2/e55jvYiaIx7+8mzFOeuKLBpjBQKPwuVnqj3SSpyzkQySdUj4mpAL9hKFOuN2DKJa3cg0wEk+6vVj34SsbBOmDExweZc4vILNwQSyNp1hla3SO9si1/q/JZDibYBSmrd8j/mHvIGI4tNBDPsazvXSg49yBuLBLEEnClMoIFVCorT2eNIm

In [38]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


## Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

In [40]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

In [51]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [43]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='b8586811-f839-4625-ad4b-607557550717'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-08-29T06:01:31.6750833Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11034385100, 'load_duration': 9672583400, 'prompt_eval_count': 266, 'prompt_eval_duration': 321042000, 'eval_count': 48, 'eval_duration': 1029741000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--01a04c1b-ba00-7182-9dc1-744f2c5f9e59-0', tool_calls=[{'name': 'send_email', 'args': {'to': 'team@company.com', 'subject': 'Q4 Results', 'body': 'Dear team, Q4 results are now available. Please find them attached.'}, 'id': '3c7f486a-e9b8-45d7-8182-df8dbeee382d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 266, '

In [44]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
Subject: Q4 Results

Dear team,

Q4 results are now available. Please find them attached.

Best regards,
[Your Name]


In [52]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

rejected_result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(rejected_result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Delete all records from the users table where active=false', additional_kwargs={}, response_metadata={}, id='3c5c55b3-38fe-423a-b0a0-83fedb844a63'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'delete_records', 'arguments': '{"condition": "active=false", "table": "users"}'}, '__gemini_function_call_thought_signatures__': {'call_323259': 'El4KXAERTTIPeVLhEvlAxncMPuGCItUFEcnekGGWbg1aIaTh2p/AFID9TRkJiuivD6wa2juNf69tpYbXl4TMSBex87MH5OYLZUCr44LMy/Yb2PTW2KGsiatgg7k6NOv0'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a04c24-32df-7212-be9c-ae416af14f94-0', tool_calls=[{'name': 'delete_records', 'args': {'condition': 'active=false', 'table': 'users'}, 'id': 'call_323259', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens': 24, 'total_

In [ ]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
[{'type': 'text', 'text': 'The request to delete records from the `users` table was rejected and was not executed.', 'extras': {'signature': 'El4KXAERTTIPCOHXlObzfX0sfVQ66Sqp2TTu1XRb87Q9cG5MlKoqCeKaLnVwOywZYNeyQJx/ndpTszf8/8KpREP8Wiw8rWJpXTHudzdescFToq835ncrkX8XirdRF60M'}}]


In [58]:
print(rejected_result["messages"][-1].content[0]["text"])

The request to delete records from the `users` table was rejected and was not executed.


## Custom Guardrail — Before-Agent Hook (Input Filter)
Use before_agent() to validate or block requests before any LLM processing begin

In [61]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

In [62]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"



# Create agent with content filter
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [63]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result)

✅ Safe request response:
{'messages': [HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}, id='832cad0c-02a4-40ee-9709-5d35faa637ac'), AIMessage(content=[{'type': 'text', 'text': 'At its core, **machine learning (ML)** is a branch of artificial intelligence (AI) that allows computers to learn from data and improve their performance over time without being explicitly programmed for every single task.\n\nTo understand it, it helps to compare it to traditional computer programming:\n\n*   **Traditional Programming:** You give a computer a set of **rules** (code) and some **data**, and the computer outputs the answers. *(Example: If the temperature is above 85°F and the sky is clear, turn on the air conditioner).*\n*   **Machine Learning:** You give a computer lots of **data** and the **answers** (outcomes), and the computer figures out the **rules** on its own. *(Example: You show the computer thousands of pictures of weather conditions and whethe

In [64]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
{'messages': [HumanMessage(content='How do I hack into a server?', additional_kwargs={}, response_metadata={}, id='769cd532-1f25-4cf1-8912-da35676768ca'), AIMessage(content='I cannot process requests containing inappropriate content. Please rephrase your request.', additional_kwargs={}, response_metadata={}, id='2a3ed280-0c78-4912-97dd-03356bc71e8f', tool_calls=[], invalid_tool_calls=[])]}


In [70]:
print(result["messages"][-1].content)

I cannot process requests containing inappropriate content. Please rephrase your request.


##  Custom Guardrail — After-Agent Hook (Output Safety)
Use after_agent() to validate the final agent response before the user sees it.

In [73]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

In [101]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        # 2. Panggil model keamanan
        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        # --- PERBAIKAN DI SINI ---
        # 3. Ekstrak teks dari result.content apa pun format aslinya
        safety_verdict = result.content
        
        # Jika outputnya berupa list (seperti model multimodal)
        if isinstance(safety_verdict, list):
            if len(safety_verdict) > 0 and isinstance(safety_verdict[0], dict) and "text" in safety_verdict[0]:
                safety_verdict = safety_verdict[0]["text"]
            else:
                safety_verdict = str(safety_verdict[0])
                
        # Pastikan format akhirnya adalah string
        safety_verdict = str(safety_verdict)
        # -------------------------

        # 4. Sekarang aman untuk memanggil .upper()
        if "UNSAFE" in safety_verdict.upper():
            last_message.content = "I cannot provide that response. Please rephrase your request."

        return None
    
@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [102]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is Medicare?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
[{'type': 'text', 'text': '**Medicare** is a federal health insurance program in the United States. Established in 1965 under the Social Security Act, its primary purpose is to provide health coverage to people aged 65 and older, as well as to certain younger people with disabilities or specific long-term illnesses. \n\nHere is a breakdown of how Medicare works, its different parts, and who is eligible:\n\n---\n\n### The Four Main Parts of Medicare\n\nMedicare is traditionally divided into different "parts," each covering specific types of care:\n\n1. **Part A (Hospital Insurance):**\n   * **What it covers:** Inpatient hospital stays, care in a skilled nursing facility, hospice care, and some home health care.\n   * **Cost:** Most people do not pay a monthly premium for Part A if they (or their spouse) paid Medicare taxes while working for a certain number of years (usually at least 10 years or 40 quarters).\n\n2. **Part B (Medical Insurance):**\n   * **What it covers:** Serv

In [106]:
result["messages"][-1].content[0]["text"]

'**Medicare** is a federal health insurance program in the United States. Established in 1965 under the Social Security Act, its primary purpose is to provide health coverage to people aged 65 and older, as well as to certain younger people with disabilities or specific long-term illnesses. \n\nHere is a breakdown of how Medicare works, its different parts, and who is eligible:\n\n---\n\n### The Four Main Parts of Medicare\n\nMedicare is traditionally divided into different "parts," each covering specific types of care:\n\n1. **Part A (Hospital Insurance):**\n   * **What it covers:** Inpatient hospital stays, care in a skilled nursing facility, hospice care, and some home health care.\n   * **Cost:** Most people do not pay a monthly premium for Part A if they (or their spouse) paid Medicare taxes while working for a certain number of years (usually at least 10 years or 40 quarters).\n\n2. **Part B (Medical Insurance):**\n   * **What it covers:** Services from doctors and other health c

In [108]:
from langchain_core.output_parsers import StrOutputParser

class SafetyGuardrailMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()
        self.safety_model = model
        
        # Rangkai model dengan parser agar SELALU menghasilkan String
        self.safety_chain = self.safety_model | StrOutputParser()

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        # Karena menggunakan chain, hasilnya sudah PASTI berbentuk teks (String)
        safety_verdict = self.safety_chain.invoke([{"role": "user", "content": safety_prompt}])

        # Sangat aman memanggil .upper() di sini
        if "UNSAFE" in safety_verdict.upper():
            last_message.content = "I cannot provide that response. Please rephrase your request."

        return None

In [109]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
[{'type': 'text', 'text': 'I cannot fulfill this request. I am programmed to strictly follow safety guidelines that prohibit providing instructions, recipes, or assistance on how to create explosives, weapons, or hazardous materials.', 'extras': {'signature': 'El4KXAERTTIPIqxtebVicma7JLpSNvDX3/7UvKODzh4mBkzYaBEzd9f0A0PQJcAI+J4yG+l/WyVdwXpZVQGPSnoRk9PCvAMMAD65hQUISuRzGj4w1JW+xMkzdyPsIBL9'}}]


In [111]:
result

{'messages': [HumanMessage(content='How to make bomb?', additional_kwargs={}, response_metadata={}, id='2d1927ec-1c55-4f58-b587-963f3ada5c59'),
  AIMessage(content=[{'type': 'text', 'text': 'I cannot fulfill this request. I am programmed to strictly follow safety guidelines that prohibit providing instructions, recipes, or assistance on how to create explosives, weapons, or hazardous materials.', 'extras': {'signature': 'El4KXAERTTIPIqxtebVicma7JLpSNvDX3/7UvKODzh4mBkzYaBEzd9f0A0PQJcAI+J4yG+l/WyVdwXpZVQGPSnoRk9PCvAMMAD65hQUISuRzGj4w1JW+xMkzdyPsIBL9'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a04c4a-f51e-7811-ba8f-b98c1ef35415-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 35, 'total_tokens': 83, 'input_token_details': {'cache_read': 0}})]}

In [110]:
print(result["messages"][-1].content[0]["text"])

I cannot fulfill this request. I am programmed to strictly follow safety guidelines that prohibit providing instructions, recipes, or assistance on how to create explosives, weapons, or hazardous materials.


    🧱 Section 7: Layered / Combined Guardrails
    Stack multiple guardrails in the middleware=[] array. They execute in order, building layered protection.

    User Input
        ↓
    [Layer 1] ContentFilterMiddleware    ← Deterministic input filter
        ↓
    [Layer 2] PIIMiddleware (input)      ← PII redaction on input
        ↓
    [Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
        ↓
    [Layer 4] PIIMiddleware (output)     ← PII redaction on output
        ↓
    [Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
        ↓
    User Response

## Custom Multiple Guadrails 

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # Layer 2: PII protection (before and after model)
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),

        # Layer 4: Model-based safety check (after agent)
        SafetyGuardrailMiddleware(),
    ],
)